# Week 43, Day 2: Grid cells: a hexagonal code and path integration

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w043/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Gridness from the autocorrelogram

Write gridness(m): compute the autocorrelogram a; build a ring mask from radius 4 bins to a quarter of the autocorrelogram's size; for each angle rotate a (scipy.ndimage.rotate, reshape=False, order=1) and correlate with a over the ring.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import rotate
from scipy.signal import fftconvolve
r = np.random.default_rng(2005)
nb = 50; g = (np.arange(nb) + 0.5) / nb; X, Y = np.meshgrid(g, g, indexing="ij")

def grid(lam=0.3, ori=0.0, phase=(0, 0)):
    k = 4 * np.pi / (np.sqrt(3) * lam); s = 0
    for a in [0, np.pi / 3, 2 * np.pi / 3]:
        s = s + np.cos(k * np.cos(a + ori) * (X - phase[0]) + k * np.sin(a + ori) * (Y - phase[1]))
    return np.maximum(0, (s + 1.5) / 4.5)

def autocorr(m):
    m = m - m.mean(); a = fftconvolve(m, m[::-1, ::-1], mode="full")
    return a / a.max()

def gridness(m):
    a = autocorr(m); n = a.shape[0]; c = n // 2
    yy, xx = np.mgrid[:n, :n]; rr = np.hypot(yy - c, xx - c)
    ring = ...
    def cc(angle):
        ...
    return min(cc(60), cc(120)) - max(cc(30), cc(90), cc(150))

maps = {"grid": grid(0.3),
        "noisy grid": grid(0.3) + 0.3 * r.normal(size=(nb, nb)),
        "place field": np.exp(-((X - 0.4) ** 2 + (Y - 0.6) ** 2) / (2 * 0.08 ** 2)),
        "square lattice": np.maximum(0, np.cos(2 * np.pi * X / 0.3) + np.cos(2 * np.pi * Y / 0.3))}
scores = {k: gridness(m) for k, m in maps.items()}
print({k: round(v, 2) for k, v in scores.items()})
fig, ax = plt.subplots(1, 2, figsize=(8, 4)); ax[0].imshow(maps["grid"].T, origin="lower"); ax[1].imshow(autocorr(maps["grid"]).T, origin="lower"); plt.show()

In [ ]:
import numpy as np
def test_grid():
    "the hexagonal map scores above 1 and stays above 1 with noise"
    assert scores["grid"] > 1 and scores["noisy grid"] > 1
def test_others():
    "a single place field scores near zero and a square lattice scores below -1"
    assert abs(scores["place field"]) < 0.2 and scores["square lattice"] < -1

test_grid()
test_others()
print("All checks passed")

### Exercise: Decoding position from three modules

Write decode(phases): for every candidate position on a 1 mm grid from 0 to 10 m, compute each module's phase (x mod period, divided by period), the circular distance to the observed phases, and return the candidate with the smallest summed squared distance.

In [ ]:
periods = np.array([0.30, 0.42, 0.59])            # metres, ratio about 1.4
xs = np.linspace(0, 10, 10001)
cand = (xs[:, None] % periods) / periods            # phases of every candidate

def decode(phases):
    d = ...
    d = np.minimum(d, 1 - d)                        # circular distance
    return ...

med, cat = {}, {}
for sd in [0.0, 0.02, 0.05, 0.1]:
    errs = []
    for k in range(200):
        x = r.uniform(0, 10)
        ph = ((x % periods) / periods + sd * r.normal(size=3)) % 1
        errs.append(abs(decode(ph) - x))
    errs = np.array(errs); med[sd] = np.median(errs); cat[sd] = np.mean(errs > 0.5)
    print(f"phase noise {sd:.2f}: median error {100 * med[sd]:.2f} cm, catastrophic errors {100 * cat[sd]:.1f}%")

In [ ]:
import numpy as np
def test_precise():
    "without phase noise, the median error is under 1 mm over a 10 m track"
    assert med[0.0] < 0.001
def test_catastrophic():
    "with phase noise of 0.05, more than 20% of decodes are off by over 0.5 m"
    assert cat[0.05] > 0.2 and cat[0.02] < 0.05

test_precise()
test_catastrophic()
print("All checks passed")

### Exercise: Errors that accumulate

Integrate 200 noisy velocity traces (noise 0.05 m/s per component) and compute the RMS position error over time. Then repeat, resetting the estimate to the true position every 500 steps (10 s), as a landmark would.

In [ ]:
dt, T = 0.02, 3000                                    # 60 s
v = 0.2 * r.normal(size=(T, 2)); x_true = np.cumsum(v * dt, 0)

def integrate(reset_every=None, noise=0.05):
    est = np.zeros(2); out = np.zeros((T, 2))
    vm = v + noise * r.normal(size=v.shape)
    for t in range(T):
        ...
        out[t] = est
    return out

errs = np.array([np.linalg.norm(integrate() - x_true, axis=1) for _ in range(200)])
errs_r = np.array([np.linalg.norm(integrate(500) - x_true, axis=1) for _ in range(200)])
rms, rms_reset = np.sqrt((errs ** 2).mean(0)), np.sqrt((errs_r ** 2).mean(0))
print(f"RMS error at 5, 20, 60 s: {np.round(100 * rms[[249, 999, 2999]], 1)} cm; with resets, never above {100 * rms_reset.max():.1f} cm")
plt.plot(np.arange(T) * dt, 100 * rms, label="integration only"); plt.plot(np.arange(T) * dt, 100 * rms_reset, label="landmark every 10 s")
plt.xlabel("time (s)"); plt.ylabel("RMS error (cm)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_sqrt():
    "RMS error at 60 s is about sqrt(12) times the error at 5 s"
    ratio = rms[2999] / rms[249]
    assert 2.8 < ratio < 4.2
def test_reset():
    "with landmark resets every 10 s, the error never exceeds half of the uncorrected error at 60 s"
    assert rms_reset.max() < 0.5 * rms[2999]

test_sqrt()
test_reset()
print("All checks passed")